# ST311 Legal Sentiment Project: ECtHR Dataset Audit and Initial EDA

This notebook is the main presentable working notebook for the project.

It covers:
- safe environment setup and authentication
- loading `AUEB-NLP/ecthr_cases` with the `violation-prediction` task
- auditing split sizes, label structure, and label frequencies
- showing readable example records
- writing a dataset audit to `docs/dataset_audit.md`
- checking document-length pressure for BERT-style models
- checking where silver rationale paragraphs appear in each case

This notebook is intentionally limited to dataset inspection and exploratory analysis. It does **not** start model training.

## 1. Environment Setup

Run this cell first in Colab. It installs the libraries needed for dataset loading, inspection, and tokenizer-based analysis.

In [5]:
# Install the notebook dependencies used for dataset loading and text analysis.
!pip install -q datasets==2.21.0 transformers huggingface_hub

python(42304) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import time

In [7]:
# Device setup:
# Prefer CUDA if running on an NVIDIA GPU, however I'm using my own IDE and don't have access to CUDA, so I will use MPS if available, otherwise CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple Silicon GPU
else:
    device = torch.device("cpu")
# Fallback to CPU if no GPU backend is available.

print("Device:", device)

Device: mps


## 2. Safe Hugging Face Authentication

This project must not use hardcoded tokens.

Preferred options:
- set an environment variable named `HF_TOKEN`
- in Colab, add a secret named `HF_TOKEN`

Authentication is optional for this public dataset and tokenizer, but keeping this cell makes the notebook safer and more portable.

In [8]:
# Resolve a Hugging Face token from the local environment or from Colab secrets.
import os
from huggingface_hub import login

try:
    # In Colab, `userdata` exposes secrets that have been added in the UI.
    from google.colab import userdata
except ImportError:
    userdata = None

# Prefer a normal environment variable, then fall back to a Colab secret.
hf_token = os.getenv("HF_TOKEN")
if not hf_token and userdata is not None:
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    login(token=hf_token)
    print("Authenticated with Hugging Face.")
else:
    print("HF_TOKEN not found. Continuing without authentication.")

HF_TOKEN not found. Continuing without authentication.


## 3. Load the ECtHR Dataset

We use the `violation-prediction` configuration from `AUEB-NLP/ecthr_cases`.

The dataset loader uses custom code, so `trust_remote_code=True` is required to avoid an interactive prompt during execution.

In [9]:
# Load the ECtHR dataset using the multi-label violation-prediction task.
from datasets import load_dataset

# `trust_remote_code=True` avoids an interactive prompt from the custom dataset loader.
dataset = load_dataset(
    "AUEB-NLP/ecthr_cases",
    "violation-prediction",
    trust_remote_code=True,
)

dataset

python(42309) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


DatasetDict({
    train: Dataset({
        features: ['facts', 'labels', 'silver_rationales'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['facts', 'labels', 'silver_rationales'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['facts', 'labels', 'silver_rationales'],
        num_rows: 1000
    })
})

## 4. Split Sizes and Feature Structure

Before doing any modelling, we need to confirm the basic shape of the dataset: how many records sit in each split and what fields are exposed by the loader.

In [11]:
# Summarize the dataset splits and inspect the feature schema exposed by the loader.
import pandas as pd

# Record the split sizes so later experiments can rely on the fixed train/validation/test setup.
split_sizes = {split_name: len(split_data) for split_name, split_data in dataset.items()}
feature_names = list(dataset["train"].features.keys())
label_feature = dataset["train"].features["labels"]

# Build a small dataframe so the split summary is easy to read in the notebook.
split_summary_df = pd.DataFrame(
    [{"split": split_name, "num_records": split_size} for split_name, split_size in split_sizes.items()]
)

print("Split sizes:", split_sizes)
print("Feature names:", feature_names)
print("Label feature:", label_feature)

split_summary_df

Split sizes: {'train': 9000, 'test': 1000, 'validation': 1000}
Feature names: ['facts', 'labels', 'silver_rationales']
Label feature: Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)


,split,num_records
0,train,9000
1,test,1000
2,validation,1000


### Why This Matters

**Motivation.** Before building any baseline, we need to know whether the dataset setup itself is clean and whether the task formulation is unambiguous.

**What the output shows.** The dataset already provides a fixed `9000 / 1000 / 1000` train-validation-test split, and the feature schema is compact: `facts`, `labels`, and `silver_rationales`. The label field is a sequence rather than a single class.

**Takeaway.** We do not need to design our own split policy, and the problem should be treated as **multi-label classification** from the start.

## 5. Label Structure and Label Frequencies

The `violation-prediction` task allows a case to contain multiple violated articles. This section inspects:
- the set of unique labels
- how many labels appear per record
- per-label frequencies across train, validation, and test

In [12]:
# Count label frequencies and label cardinality across each split.
from collections import Counter


def sort_labels(label_values):
    # Sort numeric article labels naturally while leaving protocol-style labels readable.
    def label_key(value):
        try:
            return (0, int(value))
        except ValueError:
            return (1, value)

    return sorted(label_values, key=label_key)


# Track how often each label appears and how many labels each record contains.
split_label_counters = {}
split_cardinality_counters = {}
all_labels = set()

for split_name, split_data in dataset.items():
    label_counter = Counter()
    cardinality_counter = Counter()

    for record in split_data:
        labels = record["labels"]
        label_counter.update(labels)
        cardinality_counter[len(labels)] += 1

    split_label_counters[split_name] = label_counter
    split_cardinality_counters[split_name] = cardinality_counter
    all_labels.update(label_counter.keys())

# Freeze the label order so later modelling code can reuse a consistent class list.
label_list = sort_labels(all_labels)

# Build a per-label frequency table across train, validation, and test.
label_frequency_rows = []
for label in label_list:
    label_frequency_rows.append(
        {
            "label": label,
            "train_count": split_label_counters["train"][label],
            "validation_count": split_label_counters["validation"][label],
            "test_count": split_label_counters["test"][label],
            "train_pct": round(100 * split_label_counters["train"][label] / len(dataset["train"]), 2),
        }
    )

label_frequency_df = pd.DataFrame(label_frequency_rows).sort_values(
    by=["train_count", "label"], ascending=[False, True]
).reset_index(drop=True)

# Build a table showing how many labels appear per record in each split.
cardinality_rows = []
for split_name, cardinality_counter in split_cardinality_counters.items():
    for num_labels, count in sorted(cardinality_counter.items()):
        cardinality_rows.append(
            {
                "split": split_name,
                "labels_per_record": num_labels,
                "count": count,
                "pct": round(100 * count / len(dataset[split_name]), 2),
            }
        )

cardinality_df = pd.DataFrame(cardinality_rows)

print("Unique labels:", label_list)
print("Number of unique labels:", len(label_list))
print("Train labels-per-record distribution:", dict(sorted(split_cardinality_counters["train"].items())))

label_frequency_df

Unique labels: ['2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '18', '34', '38', '46', 'P1-1', 'P1-2', 'P1-3', 'P12-1', 'P4-2', 'P4-4', 'P6-1', 'P6-3', 'P7-1', 'P7-2', 'P7-3', 'P7-4']
Number of unique labels: 29
Train labels-per-record distribution: {0: 762, 1: 5237, 2: 2348, 3: 410, 4: 211, 5: 21, 6: 10, 8: 1}


,label,train_count,validation_count,test_count,train_pct
0,6,4704,300,299,52.27
1,P1-1,1421,139,122,15.79
2,5,1368,187,166,15.20
3,3,1349,193,189,14.99
4,13,1238,88,79,13.76
5,8,710,87,123,7.89
6,2,505,57,56,5.61
7,10,291,42,77,3.23
8,14,141,18,16,1.57
9,11,110,33,37,1.22


In [13]:
# Display the labels-per-record summary as a standalone table.
cardinality_df

,split,labels_per_record,count,pct
0,train,0,762,8.47
1,train,1,5237,58.19
2,train,2,2348,26.09
3,train,3,410,4.56
4,train,4,211,2.34
5,train,5,21,0.23
6,train,6,10,0.11
7,train,8,1,0.01
8,test,0,135,13.50
9,test,1,621,62.10


### Why This Matters

**Motivation.** Label structure determines both the baseline design and the evaluation strategy, so this is one of the most important things to establish early.

**What the output shows.** The dataset contains `29` unique labels, but the distribution is highly uneven. Article `6` appears in `4704` training cases (`52.27%`), while several labels appear only `1` to `6` times. Most training cases have `1` label (`58.19%`) or `2` labels (`26.09%`), but `8.47%` have no labels at all.

**Takeaway.** Accuracy alone would be misleading here. Later experiments should emphasize `micro-F1`, `macro-F1`, and per-label reporting, and we should expect the rarest labels to remain difficult even with better models.

## 6. Readable Example Records

A few examples make the dataset structure easier to understand than summary tables alone. We inspect one record from each split in a readable format.

In [14]:
# Pull one readable example from each split so we can inspect the raw case structure.
example_specs = [("train", 0), ("validation", 0), ("test", 0)]
example_records = []

for split_name, index in example_specs:
    # Keep only a short preview of the first paragraphs to avoid flooding the notebook.
    record = dataset[split_name][index]
    example_records.append(
        {
            "split": split_name,
            "index": index,
            "labels": record["labels"],
            "num_fact_paragraphs": len(record["facts"]),
            "num_silver_rationales": len(record["silver_rationales"]),
            "paragraph_1": record["facts"][0][:280].replace("\n", " "),
            "paragraph_2": record["facts"][1][:280].replace("\n", " ") if len(record["facts"]) > 1 else "",
        }
    )

# Print the example summaries in a presentation-friendly format.
for example_record in example_records:
    print(f"Example: {example_record['split']}[{example_record['index']}]")
    print(f"Labels: {example_record['labels']}")
    print(f"Fact paragraphs: {example_record['num_fact_paragraphs']}")
    print(f"Silver rationales: {example_record['num_silver_rationales']}")
    print("Paragraph 1 preview:")
    print(example_record['paragraph_1'])
    print("Paragraph 2 preview:")
    print(example_record['paragraph_2'])
    print("-" * 80)


Example: train[0]
Labels: ['8']
Fact paragraphs: 83
Silver rationales: 19
Paragraph 1 preview:
11.  At the beginning of the events relevant to the application, K. had a daughter, P., and a son, M., born in 1986 and 1988 respectively. P.’s father is X and M.’s father is V. From March to May 1989 K. was voluntarily hospitalised for about three months, having been diagnosed a
Paragraph 2 preview:
12.  The applicants initially cohabited from the summer of 1991 to July 1993. In 1991 both P. and M. were living with them. From 1991 to 1993 K. and X were involved in a custody and access dispute concerning P. In May 1992 a residence order was made transferring custody of P. to 
--------------------------------------------------------------------------------
Example: validation[0]
Labels: []
Fact paragraphs: 7
Silver rationales: 0
Paragraph 1 preview:
5.  The applicant was born in 1983 and is detained in Sztum.
Paragraph 2 preview:
6.  At the time of the events in question, the applicant was ser

### Why This Matters

**Motivation.** Summary tables can hide what the input actually looks like, so readable examples are useful for checking that the modelling assumptions still make sense.

**What the output shows.** Case structure varies sharply. `train[0]` has `83` fact paragraphs and `19` silver rationales, `validation[0]` has only `7` fact paragraphs and no labels or rationales, and `test[0]` has `37` fact paragraphs with `26` silver rationales.

**Takeaway.** The raw input is a long sequence of legal fact paragraphs, not a short flat text classification problem. That makes document access and structure-aware preprocessing central to the project.

## 7. Write the Dataset Audit to Disk

This cell writes a short audit to `docs/dataset_audit.md` so the dataset findings are saved outside the notebook as well.

In [15]:
# Export the dataset audit to a markdown file that can be reused outside the notebook.
from pathlib import Path


def markdown_table(rows, headers):
    # Convert simple row data into a markdown table for the audit document.
    header_line = "| " + " | ".join(headers) + " |"
    separator_line = "| " + " | ".join(["---"] * len(headers)) + " |"
    body_lines = []

    for row in rows:
        cleaned_values = [str(value).replace("|", "/") for value in row]
        body_lines.append("| " + " | ".join(cleaned_values) + " |")

    return "\n".join([header_line, separator_line] + body_lines)


# Convert the in-memory summaries into markdown-ready table rows.
split_table_rows = [
    [row["split"], row["num_records"]]
    for row in split_summary_df.to_dict(orient="records")
]

label_table_rows = [
    [
        row["label"],
        row["train_count"],
        row["validation_count"],
        row["test_count"],
        row["train_pct"],
    ]
    for row in label_frequency_df.to_dict(orient="records")
]

cardinality_table_rows = [
    [row["split"], row["labels_per_record"], row["count"], row["pct"]]
    for row in cardinality_df.to_dict(orient="records")
]

# Assemble the dataset audit as a short markdown report.
audit_lines = [
    "# Dataset Audit",
    "",
    "Dataset: `AUEB-NLP/ecthr_cases`",
    "Task: `violation-prediction`",
    "",
    "## Split Sizes",
    markdown_table(split_table_rows, ["split", "num_records"]),
    "",
    "## Label Structure",
    f"- Feature names: {feature_names}",
    f"- Label feature: `{label_feature}`",
    f"- Unique labels ({len(label_list)}): {', '.join(label_list)}",
    "",
    "### Labels Per Record",
    markdown_table(cardinality_table_rows, ["split", "labels_per_record", "count", "pct"]),
    "",
    "## Label Frequencies",
    markdown_table(label_table_rows, ["label", "train_count", "validation_count", "test_count", "train_pct"]),
    "",
    "## Example Records",
]

# Append the readable example records to the end of the audit.
for example_record in example_records:
    audit_lines.extend(
        [
            f"### {example_record['split']}[{example_record['index']}]",
            f"- Labels: {example_record['labels']}",
            f"- Fact paragraphs: {example_record['num_fact_paragraphs']}",
            f"- Silver rationales: {example_record['num_silver_rationales']}",
            f"- Paragraph 1: {example_record['paragraph_1']}",
            f"- Paragraph 2: {example_record['paragraph_2']}",
            "",
        ]
    )

# Ensure the docs directory exists before writing the audit artifact.
Path("docs").mkdir(exist_ok=True)
audit_path = Path("docs/dataset_audit.md")
audit_path.write_text("\n".join(audit_lines))

print(f"Wrote dataset audit to {audit_path}")


Wrote dataset audit to docs/dataset_audit.md


### Why This Matters

**Motivation.** Key dataset facts should not live only inside an executed notebook, especially when the project write-up and group coordination will depend on them later.

**What the output shows.** The audit has been written to `docs/dataset_audit.md`, so the current dataset structure, label summary, and example records are now stored as a reusable project artifact.

**Takeaway.** We now have a stable dataset reference for the report and for later experiment sections, without needing to repeat the audit logic elsewhere.

## 8. Document Length Analysis

A key design question for the project is whether standard BERT truncation discards too much of the case text. This cell measures token length using the `bert-base-uncased` tokenizer.

In [16]:
# Estimate how often full ECtHR cases exceed the standard 512-token BERT limit.
import numpy as np
from transformers import AutoTokenizer

# Use a standard BERT tokenizer to estimate how often ECtHR cases fit within 512 tokens.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize each full case without truncation so we can measure the true input length.
lengths = []
for case in dataset["train"]:
    full_text = " ".join(case["facts"])
    token_ids = tokenizer(full_text, truncation=False)["input_ids"]
    lengths.append(len(token_ids))

lengths = np.array(lengths)

print(f"Mean token length: {np.mean(lengths):.0f}")
print(f"Median token length: {np.median(lengths):.0f}")
print(f"25th percentile: {np.percentile(lengths, 25):.0f}")
print(f"75th percentile: {np.percentile(lengths, 75):.0f}")
print(f"Percent fitting in 512 tokens: {np.mean(lengths <= 512) * 100:.1f}%")
print(f"Maximum token length: {np.max(lengths)}")


Token indices sequence length is longer than the specified maximum sequence length for this model (8507 > 512). Running this sequence through the model will result in indexing errors


Mean token length: 2040
Median token length: 1226
25th percentile: 611
75th percentile: 2475
Percent fitting in 512 tokens: 19.5%
Maximum token length: 46714


### Why This Matters

**Motivation.** The core paper claim depends on long-document pressure being real, so the notebook needs to establish that directly rather than assume it.

**What the output shows.** The mean training-case length is `2040` tokens, the median is `1226`, and even the `25th` percentile is `611`, which is already above the `512`-token BERT limit. Only `19.5%` of cases fit within `512` tokens, and the maximum length reaches `46714` tokens.

**Takeaway.** This is unequivocally a long-document problem. Head truncation will discard information for most cases, so comparing `head` against `head+tail` is a central experiment rather than an optional refinement.

## 9. Silver Rationale Position Analysis

The silver rationale annotations identify paragraphs cited by the court. If these paragraphs tend to appear late in a document, head truncation may systematically remove legally relevant content.

In [17]:
# Measure where court-cited rationale paragraphs appear within each document.
rationale_positions = []

for case in dataset["train"]:
    # Convert paragraph indices into positions relative to the full document length.
    num_paragraphs = len(case["facts"])
    for rationale_idx in case["silver_rationales"]:
        rationale_positions.append(rationale_idx / num_paragraphs)

rationale_positions = np.array(rationale_positions)

print(f"Mean relative position of rationale paragraphs: {np.mean(rationale_positions):.2f}")
print(f"Percent of rationale paragraphs in the first 25% of the document: {np.mean(rationale_positions < 0.25) * 100:.1f}%")
print(f"Percent of rationale paragraphs in the first 50% of the document: {np.mean(rationale_positions < 0.50) * 100:.1f}%")


Mean relative position of rationale paragraphs: 0.54
Percent of rationale paragraphs in the first 25% of the document: 18.1%
Percent of rationale paragraphs in the first 50% of the document: 43.7%


### Why This Matters

**Motivation.** Token counts alone do not tell us whether truncation removes legally important content, so we also need a content-based proxy.

**What the output shows.** The average relative rationale position is `0.54`, only `18.1%` of rationale paragraphs lie in the first quarter of the document, and only `43.7%` lie in the first half.

**Takeaway.** The cited evidence is usually not concentrated near the start of the document. That gives a direct legal-content reason to distrust pure head truncation and strengthens the case for comparing `head` with `head+tail`.

## 10. Takeaways for the Main Project

**Motivation.** The purpose of the EDA is not just to describe the dataset; it is to narrow the modelling problem into something defensible and executable.

**What the output shows.** The task is clearly multi-label, the label distribution is highly imbalanced, long-document pressure is severe, and court-cited rationale paragraphs are often located beyond the start of the document.

**Takeaway.** These results justify a project centred on multi-label ECHR violation prediction under long-document constraints, with careful metric choice and simple baselines before more ambitious models.

## 11. TF-IDF + OvR Logistic Regression Baseline

The first experimental system is the simplest full-text baseline in the project plan: TF-IDF features over concatenated fact text, followed by a One-vs-Rest logistic regression classifier.

This baseline matters because it gives us a cheap, reproducible reference point before moving to transformer models.

In [18]:
# Train the planned full-text sparse baseline and save its outputs to disk.
import json
import time
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# Create a fixed run directory so later comparisons use the same naming convention.
run_id = "tfidf_ovr_fulltext"
run_dir = Path("results") / run_id
run_dir.mkdir(parents=True, exist_ok=True)

def join_facts(split_data):
    # Flatten each case into a single full-text string for the sparse baseline.
    return [" ".join(record["facts"]) for record in split_data]

# Build text inputs and multi-label targets for train, validation, and test.
X_train_text = join_facts(dataset["train"])
X_val_text = join_facts(dataset["validation"])
X_test_text = join_facts(dataset["test"])

y_train_labels = [record["labels"] for record in dataset["train"]]
y_val_labels = [record["labels"] for record in dataset["validation"]]
y_test_labels = [record["labels"] for record in dataset["test"]]

# Convert the string label sets into a fixed binary matrix representation.
mlb = MultiLabelBinarizer(classes=label_list)
y_train = mlb.fit_transform(y_train_labels)
y_val = mlb.transform(y_val_labels)
y_test = mlb.transform(y_test_labels)

# Use TF-IDF over unigrams and bigrams as the feature representation.
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
)

# Train one logistic regression classifier per label in an OvR setup.
classifier = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        random_state=42,
    )
)

# Fit the vectorizer and classifier, then record the total training time.
start_time = time.time()
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)
classifier.fit(X_train, y_train)
runtime_seconds = time.time() - start_time

# Generate validation and test predictions for the main reported metrics.
val_pred = classifier.predict(X_val)
test_pred = classifier.predict(X_test)

# Restrict detailed per-label reporting to labels with enough training support.
frequent_label_threshold = 50
frequent_labels = [label for label in label_list if split_label_counters["train"][label] >= frequent_label_threshold]

# Compute per-label F1 scores for the frequent labels listed in the experiment plan.
per_label_f1_rows = []
for label in frequent_labels:
    label_idx = mlb.classes_.tolist().index(label)
    per_label_f1_rows.append(
        {
            "label": label,
            "train_count": int(split_label_counters["train"][label]),
            "test_f1": float(f1_score(y_test[:, label_idx], test_pred[:, label_idx], zero_division=0)),
        }
    )

per_label_f1_df = pd.DataFrame(per_label_f1_rows).sort_values(
    by=["train_count", "label"], ascending=[False, True]
).reset_index(drop=True)

# Collect the headline metrics needed for the experiment table.
metrics = {
    "run_id": run_id,
    "model": "tfidf_ovr_logreg",
    "document_access": "full_text",
    "val_micro_f1": float(f1_score(y_val, val_pred, average="micro", zero_division=0)),
    "test_micro_f1": float(f1_score(y_test, test_pred, average="micro", zero_division=0)),
    "test_macro_f1": float(f1_score(y_test, test_pred, average="macro", zero_division=0)),
    "test_subset_accuracy": float(accuracy_score(y_test, test_pred)),
    "runtime_seconds": float(runtime_seconds),
    "frequent_label_threshold": frequent_label_threshold,
}

# Full-text access sees the whole document, so rationale coverage is 1.0 by construction.
rationale_coverage = {
    "run_id": run_id,
    "document_access": "full_text",
    "train_mean_coverage": 1.0,
    "validation_mean_coverage": 1.0,
    "test_mean_coverage": 1.0,
}

def decode_predictions(binary_matrix):
    # Convert predicted binary vectors back into human-readable label lists.
    return [list(labels) for labels in mlb.inverse_transform(binary_matrix)]

# Save validation and test predictions in a readable CSV format.
val_predictions_df = pd.DataFrame(
    {
        "case_index": list(range(len(y_val_labels))),
        "true_labels": ["|".join(labels) for labels in y_val_labels],
        "pred_labels": ["|".join(labels) for labels in decode_predictions(val_pred)],
    }
)

test_predictions_df = pd.DataFrame(
    {
        "case_index": list(range(len(y_test_labels))),
        "true_labels": ["|".join(labels) for labels in y_test_labels],
        "pred_labels": ["|".join(labels) for labels in decode_predictions(test_pred)],
    }
)

# Save the run configuration so the baseline is reproducible later.
config = {
    "run_id": run_id,
    "task": "violation-prediction",
    "model": "tfidf_ovr_logreg",
    "document_access": "full_text",
    "vectorizer": {
        "max_features": 50000,
        "ngram_range": [1, 2],
        "min_df": 2,
        "stop_words": "english",
    },
    "classifier": {
        "base_estimator": "LogisticRegression",
        "solver": "liblinear",
        "max_iter": 1000,
        "random_state": 42,
    },
    "frequent_label_threshold": frequent_label_threshold,
}

# Write all baseline artifacts to the expected results directory.
(run_dir / "config.json").write_text(json.dumps(config, indent=2))
(run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
(run_dir / "rationale_coverage.json").write_text(json.dumps(rationale_coverage, indent=2))
val_predictions_df.to_csv(run_dir / "val_predictions.csv", index=False)
test_predictions_df.to_csv(run_dir / "test_predictions.csv", index=False)
per_label_f1_df.to_csv(run_dir / "per_label_f1.csv", index=False)

# Format the key metrics for display in the notebook.
metrics_df = pd.DataFrame(
    [
        {
            "val_micro_f1": round(metrics["val_micro_f1"], 4),
            "test_micro_f1": round(metrics["test_micro_f1"], 4),
            "test_macro_f1": round(metrics["test_macro_f1"], 4),
            "test_subset_accuracy": round(metrics["test_subset_accuracy"], 4),
            "runtime_seconds": round(metrics["runtime_seconds"], 2),
            "rationale_coverage": rationale_coverage["test_mean_coverage"],
        }
    ]
)

# Show the main run summary and the detailed frequent-label scores.
print(f"Saved run outputs to {run_dir}")
display(metrics_df)
display(per_label_f1_df)


Saved run outputs to results/tfidf_ovr_fulltext


,val_micro_f1,test_micro_f1,test_macro_f1,test_subset_accuracy,runtime_seconds,rationale_coverage
0,0.6018,0.5795,0.1394,0.406,105.23,1.0


,label,train_count,test_f1
0,6,4704,0.726003
1,P1-1,1421,0.603015
2,5,1368,0.691176
3,3,1349,0.753541
4,13,1238,0.250000
5,8,710,0.149660
6,2,505,0.697674
7,10,291,0.117647
8,14,141,0.000000
9,11,110,0.052632


### What This Output Means

**Motivation.** The TF-IDF baseline gives us the cheapest full-text benchmark in the project. If a simple sparse baseline is already competitive, later transformer gains need to justify their additional complexity.

**What the output shows.** The most important headline number here is the `test_micro_f1` of `0.5795`, because micro-F1 is the most natural aggregate metric for this imbalanced multi-label setting and gives the baseline we need to beat overall. The `val_micro_f1` of `0.6018` is reasonably close, so there is no obvious sign of severe overfitting. The `test_macro_f1` is much lower at `0.1394`, which tells us that performance collapses once we average across labels more equally instead of letting the frequent labels dominate. The `subset_accuracy` of `0.4060` is useful to report, but it is a stricter metric and less informative than micro-F1 here because a prediction is counted as wrong unless the full label set is exactly correct.

**What the label-level results show.** Some frequent labels are already predicted fairly well by sparse lexical features: label `3` reaches `0.7535`, label `6` reaches `0.7260`, label `2` reaches `0.6977`, label `5` reaches `0.6912`, and `P1-1` reaches `0.6030`. A strong F1 for a label like this means that, for that article, the wording patterns in the full facts already contain a lot of predictive signal and a simple full-text bag-of-ngrams model can exploit it. In contrast, labels such as `13` (`0.2500`), `8` (`0.1497`), `10` (`0.1176`), `11` (`0.0526`), `14` (`0.0000`), and `34` (`0.0000`) are much weaker, which suggests a mix of label rarity, harder semantics, and cases where simple lexical matching is not enough.

**Takeaway.** This is a serious baseline, not a formality. Because it uses the full text, it is especially important as a reference point for the later transformer runs. For the paper, the most convincing improvement would be a transformer model that beats `0.5795` on test micro-F1 **and** improves meaningfully on macro-F1 over `0.1394`, especially by lifting the weaker medium-frequency labels rather than only matching the already-strong frequent ones. We should not assume that every transformer will beat this baseline: a head-truncated BERT model could easily lose to TF-IDF because it sees less of the document. The strongest evidence for the paper's claim would be if `head+tail`, and ideally Legal-BERT with `head+tail`, outperform this baseline in overall micro-F1 while also showing better balance across labels.

## 12. BERT-Based Models: Experimental Design

**Motivation.** The TF-IDF baseline establishes what is achievable with a simple bag-of-words representation over the full document. The next question is whether a pretrained transformer can do better — and if so, under what conditions. Two factors are varied systematically: the encoder's pretraining domain and the portion of the document it can access.

### Models

Two encoders are compared, forming a progression in domain specificity:

- **`bert-base-uncased`** — standard BERT pretrained on general English text (Wikipedia + BooksCorpus). No exposure to legal language during pretraining.
- **`nlpaueb/bert-base-uncased-echr`** — Legal-BERT further pretrained on ECHR-specific case documents (Chalkidis et al., 2020). The model has seen the same type of text it is being asked to classify.

Both share the same architecture (12 transformer layers, 110M parameters, identical tokenizer vocabulary), so any performance difference is attributable to pretraining data alone.

### Truncation Strategies

BERT's 512-token context limit means most cases must be truncated — only 19.5% of cases in this dataset fit within the limit. Two strategies are evaluated:

- **Head truncation** — first 512 tokens. The default in all prior ECHR work; our replication baseline. Given that court-cited rationale paragraphs sit at mean relative document position 0.54, this strategy systematically discards the majority of legally relevant content.
- **Head+tail truncation** — first 255 and last 255 content tokens (Sun et al., 2019). Designed to capture both the opening context and the later paragraphs where rationales tend to appear.

### Experiment Grid

This gives four runs in a 2×2 design:

| run_id | encoder | document access |
|---|---|---|
| `bert_head` | `bert-base-uncased` | first 512 tokens |
| `legalbert_head` | `nlpaueb/bert-base-uncased-echr` | first 512 tokens |
| `bert_head_tail` | `bert-base-uncased` | 255 head + 255 tail tokens |
| `legalbert_head_tail` | `nlpaueb/bert-base-uncased-echr` | 255 head + 255 tail tokens |

### Evaluation

All runs are evaluated on the held-out test set using micro-F1 (primary), macro-F1, and subset accuracy. Per-label F1 is reported for frequent labels only (≥ 50 training examples). Rationale coverage — the fraction of court-cited paragraphs falling within the token window — is recorded as a descriptive measure of document access quality for each strategy.

## 13. Full BERT Runs — Head Truncation

Head truncation keeps the **first 512 tokens** of each concatenated fact sequence. This is the default strategy in all prior ECHR work (Chalkidis et al., 2019; 2021) and serves as our replication baseline.

We run two models under this strategy:

| run_id | model | document access |
|---|---|---|
| `bert_head` | `bert-base-uncased` | first 512 tokens |
| `legalbert_head` | `nlpaueb/bert-base-uncased-echr` | first 512 tokens |

Comparing these two models under identical conditions isolates the effect of **legal domain pretraining** while holding the input strategy constant.

In [ ]:
# ── Hyperparameters and device setup ────────────────────────────────────────
# All four full BERT runs share these settings so comparisons are controlled.
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

FULL_MAX_LENGTH = 512       # BERT's hard token limit
FULL_BATCH_SIZE = 16        # safe for T4 VRAM with bert-base-uncased
FULL_LEARNING_RATE = 2e-5   # standard BERT fine-tuning LR (Sun et al., 2019)
FULL_NUM_EPOCHS = 3         # enough to converge on 9 k training examples
FULL_SEED = 42
FREQUENT_LABEL_THRESHOLD = 50  # minimum training occurrences to count as "frequent"

torch.manual_seed(FULL_SEED)

# Full BERT runs require a GPU; warn loudly if none is found.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: no GPU detected — full BERT runs will be extremely slow on CPU.")
print(f"Training device: {DEVICE}")

# Identify labels with enough training examples to make per-label F1 meaningful.
_label_counts = Counter(lbl for rec in dataset["train"] for lbl in rec["labels"])
frequent_labels = sorted(
    [lbl for lbl, cnt in _label_counts.items() if cnt >= FREQUENT_LABEL_THRESHOLD]
)
print(f"Frequent labels (n={len(frequent_labels)}): {frequent_labels}")

In [ ]:
# ── Dataset class, text/label helpers, and tokenization strategies ───────────

class MultiLabelTorchDataset(Dataset):
    """Wraps tokenized encodings and multi-hot label vectors for PyTorch."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Return one dict of tensors; keys must match what the model expects.
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


def encode_label_sets(records, ordered_labels):
    """Convert each record's string label list into a fixed-length multi-hot vector."""
    label_to_idx = {lbl: i for i, lbl in enumerate(ordered_labels)}
    result = []
    for rec in records:
        vec = [0.0] * len(ordered_labels)
        for lbl in rec["labels"]:
            vec[label_to_idx[lbl]] = 1.0
        result.append(vec)
    return result


def join_fact_text(records):
    """Flatten each case's fact paragraphs into a single whitespace-joined string."""
    return [" ".join(rec["facts"]) for rec in records]


def move_batch_to_device(batch, device):
    """Move every tensor in a minibatch dict to the target device."""
    return {k: v.to(device) for k, v in batch.items()}


def head_tokenize(texts, tokenizer):
    """
    Head truncation: keep the first FULL_MAX_LENGTH tokens.
    The HuggingFace tokenizer handles this natively with truncation=True.
    This is the default strategy used in all prior ECHR work.
    """
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=FULL_MAX_LENGTH,
    )


def head_tail_tokenize(texts, tokenizer):
    """
    Head+tail truncation (Sun et al., 2019): keep the first 255 and last 255
    content tokens, bookended by [CLS] and [SEP], for a total of 512 tokens.
    For texts short enough to fit without truncation, no tokens are dropped.
    """
    # Reserve 1 slot each for [CLS] and [SEP]; split the remaining 510 evenly.
    head_content = (FULL_MAX_LENGTH // 2) - 1   # 255 tokens from the start
    tail_content = (FULL_MAX_LENGTH // 2) - 1   # 255 tokens from the end

    all_input_ids, all_attention_masks, all_token_type_ids = [], [], []

    for text in texts:
        # Encode without special tokens to count raw content token IDs.
        token_ids = tokenizer.encode(text, add_special_tokens=False)

        if len(token_ids) <= (FULL_MAX_LENGTH - 2):
            # Text fits within the limit — no truncation needed.
            content_ids = token_ids
        else:
            # Concatenate head and tail slices.
            content_ids = token_ids[:head_content] + token_ids[-tail_content:]

        # Reconstruct the full BERT input: [CLS] + content + [SEP] + padding.
        input_ids = [tokenizer.cls_token_id] + content_ids + [tokenizer.sep_token_id]
        attn_mask = [1] * len(input_ids)
        type_ids  = [0] * len(input_ids)

        pad_len    = FULL_MAX_LENGTH - len(input_ids)
        input_ids += [tokenizer.pad_token_id] * pad_len
        attn_mask += [0] * pad_len
        type_ids  += [0] * pad_len

        all_input_ids.append(input_ids)
        all_attention_masks.append(attn_mask)
        all_token_type_ids.append(type_ids)

    return {
        "input_ids":      all_input_ids,
        "attention_mask": all_attention_masks,
        "token_type_ids": all_token_type_ids,
    }

In [ ]:
# ── Evaluation, rationale coverage, and artifact-saving helpers ──────────────

def evaluate_model(model, dataloader, device):
    """
    Run inference over a dataloader.
    Returns micro-F1, macro-F1, subset accuracy, and raw prediction/reference arrays.
    """
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = move_batch_to_device(batch, device)
            all_logits.append(model(**batch).logits.detach().cpu())
            all_labels.append(batch["labels"].detach().cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    # Threshold sigmoid outputs at 0.5 to obtain binary predictions.
    preds = (torch.sigmoid(logits) >= 0.5).int().numpy()
    refs  = labels.int().numpy()

    return {
        "micro_f1":        float(f1_score(refs, preds, average="micro", zero_division=0)),
        "macro_f1":        float(f1_score(refs, preds, average="macro", zero_division=0)),
        "subset_accuracy": float(accuracy_score(refs, preds)),
        "predictions":     preds,
        "references":      refs,
    }


def compute_per_label_f1(predictions, references, label_list, frequent_labels):
    """Return a DataFrame of test-set F1 scores for frequent labels only."""
    per_label = f1_score(references, predictions, average=None, zero_division=0)
    rows = [
        {"label": lbl, "test_f1": round(float(per_label[label_list.index(lbl)]), 4)}
        for lbl in frequent_labels
    ]
    return pd.DataFrame(rows).sort_values("test_f1", ascending=False)


def save_predictions_csv(predictions, references, label_list, path):
    """Save binary prediction and true-label arrays side-by-side as a CSV."""
    pred_df = pd.DataFrame(predictions, columns=[f"pred_{l}" for l in label_list])
    ref_df  = pd.DataFrame(references,  columns=[f"true_{l}" for l in label_list])
    pd.concat([pred_df, ref_df], axis=1).to_csv(path, index=False)


def compute_rationale_coverage(split, tokenizer, strategy):
    """
    Compute the mean fraction of silver-rationale paragraphs that fall inside
    the model's token window, averaged over cases that have at least one rationale.

    A paragraph is treated as covered if its cumulative token count fits within
    the window budget — either head-only (512 tokens) or head+tail (255 each side).
    Cases with no rationale annotations are skipped.
    """
    head_budget = (FULL_MAX_LENGTH // 2) - 1
    tail_budget = (FULL_MAX_LENGTH // 2) - 1

    coverages = []
    for rec in split:
        silver = list(rec["silver_rationales"])
        if not silver:
            continue

        facts = rec["facts"]
        # Pre-compute token length of each paragraph (no special tokens).
        para_lengths = [
            len(tokenizer.encode(p, add_special_tokens=False)) for p in facts
        ]
        included = set()

        if strategy == "head":
            budget, used = FULL_MAX_LENGTH - 2, 0
            for i, length in enumerate(para_lengths):
                if used + length <= budget:
                    included.add(i)
                    used += length
                else:
                    break  # paragraphs are contiguous so stop at first miss

        elif strategy == "head_tail":
            # Head pass: accumulate from the beginning of the document.
            used = 0
            for i, length in enumerate(para_lengths):
                if used + length <= head_budget:
                    included.add(i)
                    used += length
                else:
                    break
            # Tail pass: accumulate from the end of the document.
            used = 0
            for i in range(len(para_lengths) - 1, -1, -1):
                if used + para_lengths[i] <= tail_budget:
                    included.add(i)
                    used += para_lengths[i]
                else:
                    break

        covered = sum(1 for r in silver if r in included)
        coverages.append(covered / len(silver))

    return float(np.mean(coverages)) if coverages else 0.0

In [ ]:
# ── run_bert_experiment: full fine-tuning loop ───────────────────────────────

def run_bert_experiment(run_id, model_name, document_access, tokenize_fn):
    """
    Fine-tune a BERT-based model for multi-label ECHR violation prediction.

    Trains for FULL_NUM_EPOCHS epochs, selects the best checkpoint by validation
    micro-F1, then evaluates on the test set. All artifacts are written to
    results/<run_id>/ following the project output convention.

    Parameters
    ----------
    run_id           : unique identifier, e.g. 'bert_head'
    model_name       : HuggingFace model identifier
    document_access  : 'head' or 'head_tail' — used in metadata and coverage computation
    tokenize_fn      : head_tokenize or head_tail_tokenize

    Returns
    -------
    metrics      : dict of saved metric values
    per_label_df : DataFrame of frequent-label F1 scores
    """
    run_dir = Path("results") / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(FULL_SEED)

    # Persist the run configuration for reproducibility.
    config = {
        "run_id": run_id, "model_name": model_name,
        "document_access": document_access, "max_length": FULL_MAX_LENGTH,
        "batch_size": FULL_BATCH_SIZE, "learning_rate": FULL_LEARNING_RATE,
        "num_epochs": FULL_NUM_EPOCHS, "seed": FULL_SEED,
        "task": "violation-prediction",
    }
    (run_dir / "config.json").write_text(json.dumps(config, indent=2))

    # Tokenize all three splits using the chosen strategy.
    print(f"[{run_id}] Tokenizing ({document_access})...")
    tokenizer       = AutoTokenizer.from_pretrained(model_name)
    train_encodings = tokenize_fn(join_fact_text(dataset["train"]),      tokenizer)
    val_encodings   = tokenize_fn(join_fact_text(dataset["validation"]), tokenizer)
    test_encodings  = tokenize_fn(join_fact_text(dataset["test"]),       tokenizer)

    train_labels = encode_label_sets(dataset["train"],      label_list)
    val_labels   = encode_label_sets(dataset["validation"], label_list)
    test_labels  = encode_label_sets(dataset["test"],       label_list)

    train_loader = DataLoader(
        MultiLabelTorchDataset(train_encodings, train_labels),
        batch_size=FULL_BATCH_SIZE, shuffle=True,
    )
    val_loader = DataLoader(
        MultiLabelTorchDataset(val_encodings, val_labels),
        batch_size=FULL_BATCH_SIZE, shuffle=False,
    )
    test_loader = DataLoader(
        MultiLabelTorchDataset(test_encodings, test_labels),
        batch_size=FULL_BATCH_SIZE, shuffle=False,
    )

    # Load the pre-trained model with a fresh multi-label classification head.
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        problem_type="multi_label_classification",
    )
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=FULL_LEARNING_RATE)

    # Training loop: checkpoint the epoch with the best validation micro-F1.
    best_val_f1, best_state = -1.0, None
    run_start = time.time()

    for epoch in range(FULL_NUM_EPOCHS):
        model.train()
        losses = []
        for batch in train_loader:
            batch = move_batch_to_device(batch, DEVICE)
            optimizer.zero_grad()
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        val_res = evaluate_model(model, val_loader, DEVICE)
        print(
            f"[{run_id}] Epoch {epoch+1}/{FULL_NUM_EPOCHS} "
            f"— loss: {np.mean(losses):.4f} | val micro-F1: {val_res['micro_f1']:.4f}"
        )
        if val_res["micro_f1"] > best_val_f1:
            best_val_f1 = val_res["micro_f1"]
            # Clone state dict so later updates don't overwrite the best checkpoint.
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    runtime_seconds = time.time() - run_start

    # Restore best checkpoint and run final evaluation on val and test.
    model.load_state_dict(best_state)
    val_res  = evaluate_model(model, val_loader,  DEVICE)
    test_res = evaluate_model(model, test_loader, DEVICE)

    # Rationale coverage is a property of the tokenization strategy, not the model
    # weights, so values will be identical for runs sharing the same strategy.
    print(f"[{run_id}] Computing rationale coverage (may take a few minutes)...")
    coverage = {
        "run_id": run_id, "document_access": document_access,
        "train_mean_coverage":      compute_rationale_coverage(dataset["train"],      tokenizer, document_access),
        "validation_mean_coverage": compute_rationale_coverage(dataset["validation"], tokenizer, document_access),
        "test_mean_coverage":       compute_rationale_coverage(dataset["test"],       tokenizer, document_access),
    }
    (run_dir / "rationale_coverage.json").write_text(json.dumps(coverage, indent=2))

    # Compile and save all required artifacts.
    metrics = {
        "run_id": run_id, "model": model_name, "document_access": document_access,
        "val_micro_f1":             val_res["micro_f1"],
        "test_micro_f1":            test_res["micro_f1"],
        "test_macro_f1":            test_res["macro_f1"],
        "test_subset_accuracy":     test_res["subset_accuracy"],
        "runtime_seconds":          runtime_seconds,
        "frequent_label_threshold": FREQUENT_LABEL_THRESHOLD,
    }
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

    save_predictions_csv(val_res["predictions"],  val_res["references"],  label_list, run_dir / "val_predictions.csv")
    save_predictions_csv(test_res["predictions"], test_res["references"], label_list, run_dir / "test_predictions.csv")

    per_label_df = compute_per_label_f1(
        test_res["predictions"], test_res["references"], label_list, frequent_labels
    )
    per_label_df.to_csv(run_dir / "per_label_f1.csv", index=False)

    print(f"\n[{run_id}] Done in {runtime_seconds/60:.1f} min. Artifacts saved to {run_dir}/")
    print(f"  val micro-F1:    {metrics['val_micro_f1']:.4f}")
    print(f"  test micro-F1:   {metrics['test_micro_f1']:.4f}")
    print(f"  test macro-F1:   {metrics['test_macro_f1']:.4f}")
    print(f"  subset accuracy: {metrics['test_subset_accuracy']:.4f}")

    return metrics, per_label_df

In [ ]:
# Run bert_head: bert-base-uncased with head truncation.
# This is the direct replication of the strategy used in Chalkidis et al. (2019).


# Expected runtime on a T4 GPU: ~25-35 minutes.
bert_head_metrics, bert_head_per_label = run_bert_experiment(
    run_id="bert_head",
    model_name="bert-base-uncased",
    document_access="head",
    tokenize_fn=head_tokenize,
)
display(bert_head_per_label)

In [ ]:
# Run legalbert_head: Legal-BERT (ECHR sub-domain) with head truncation.
# Same strategy as bert_head; only the encoder weights differ.
# Comparing the two isolates the effect of legal domain pretraining.
# Expected runtime on a T4 GPU: ~25-35 minutes.
legalbert_head_metrics, legalbert_head_per_label = run_bert_experiment(
    run_id="legalbert_head",
    model_name="nlpaueb/bert-base-uncased-echr",
    document_access="head",
    tokenize_fn=head_tokenize,
)
display(legalbert_head_per_label)

### What This Output Means

**Motivation.** These two runs establish the head-truncation baseline for both encoder types. Because both models see identical token sequences, any performance difference is attributable solely to their pretraining corpora.

**What the output shows.** Per-label F1 scores for the frequent labels. Labels with high individual F1 (e.g. Article 6, which appears in over 52% of training cases) are expected to score well; rare labels will remain near zero regardless of model quality, which is why macro-F1 is typically much lower than micro-F1 on this dataset.

**Takeaway.** If Legal-BERT outperforms BERT here, domain pretraining helps even when both models have the same (limited) document access. This result will be revisited after the head+tail runs to test whether the gap changes when more of the document is visible.

*This section will be updated with observed numbers once the runs complete.*

## 14. Full BERT Runs — Head+Tail Truncation

Head+tail truncation takes the **first 255 tokens** and **last 255 tokens** of each document, separated by `[CLS]` and `[SEP]`, for a total of 512 tokens (Sun et al., 2019). Because court-cited rationale paragraphs sit at mean relative position 0.54 in this dataset, this strategy is expected to include more legally relevant content than head truncation alone.

We run both models under this strategy:

| run_id | model | document access |
|---|---|---|
| `bert_head_tail` | `bert-base-uncased` | 255 head + 255 tail tokens |
| `legalbert_head_tail` | `nlpaueb/bert-base-uncased-echr` | 255 head + 255 tail tokens |

The `head_tail_tokenize` function defined in Section 13 handles the custom truncation logic. All other hyperparameters are identical to the head-truncation runs, so the comparison is controlled.

In [ ]:
# Run bert_head_tail: bert-base-uncased with head+tail truncation.
# The first 255 and last 255 content tokens are concatenated, giving the model
# access to both the opening context and the later paragraphs where rationales tend to sit.
# Expected runtime on a T4 GPU: ~25-35 minutes.
bert_head_tail_metrics, bert_head_tail_per_label = run_bert_experiment(
    run_id="bert_head_tail",
    model_name="bert-base-uncased",
    document_access="head_tail",
    tokenize_fn=head_tail_tokenize,
)
display(bert_head_tail_per_label)

In [ ]:
# Run legalbert_head_tail: Legal-BERT (ECHR sub-domain) with head+tail truncation.
# This is the main comparison of interest: does the combination of domain pretraining
# and improved document access produce the best results?
# Expected runtime on a T4 GPU: ~25-35 minutes.
legalbert_head_tail_metrics, legalbert_head_tail_per_label = run_bert_experiment(
    run_id="legalbert_head_tail",
    model_name="nlpaueb/bert-base-uncased-echr",
    document_access="head_tail",
    tokenize_fn=head_tail_tokenize,
)
display(legalbert_head_tail_per_label)

### What This Output Means

**Motivation.** These runs test whether giving the model access to the tail of the document — where court-cited rationale paragraphs are concentrated — translates into better violation prediction. This is the central empirical claim of the paper.

**What the output shows.** Per-label F1 for frequent labels, comparable directly to the head-truncation results above. The rationale coverage figures saved to `results/<run_id>/rationale_coverage.json` quantify how much more of the cited evidence each strategy exposes.

**Takeaway.** Three result patterns matter most for the paper's argument:
- If `head_tail` consistently outperforms `head` for both models, the truncation strategy matters and the paper's main claim holds.
- If Legal-BERT gains more from head+tail than BERT does, domain pretraining and document access interact.
- If the coverage improvement doesn't map to any performance improvement, the rationale-position motivation is weakened.

*This section will be updated with observed numbers once the runs complete.*